# Time Series Transformer for P(RUL > 2 days) Classification

**Author:** [Your name]

**Task:** Binary classification. Predict whether Remaining Useful Life of an aircraft part exceeds 2 days, derived from `before_after` and `date_diff`.

**Model:** Transformer Encoder (PyTorch)

**Status:** Pipeline scaffold using 18 real flights from the NGAFID dataset (9 before, 9 after). Replace the assumption block with the full preprocessed dataset when feature engineering is complete.

---

## Pipeline Overview

1. Assumptions and Real Data Sampling (temporary, replace with full preprocessed data)
2. Data Preparation: Dataset, DataLoader, attention masks
3. Model Architecture: Transformer encoder
4. Training Setup: loss, optimizer, scheduler, metrics
5. Training Loop: early stopping, checkpointing
6. Evaluation: test metrics, confusion matrix, attention visualization
7. Ablation Studies (optional experiments)
8. Documentation and Handoff


---
## Step 0. Assumption Block (Temporary)

**Purpose:** This block loads a small sample (18 flights) of real sensor data from the NGAFID dataset for pipeline testing. When the feature engineering teammate finishes the full preprocessing, **only this block needs to be replaced**. Everything downstream consumes the variables defined here.

**Assumptions being made:**

* Only 2 classes used: `before` and `after`. No `same`.
* Target derived from `before_after` and `date_diff`. Label 0 = `before` (part will fail soon). Label 1 = `after` (part is healthy).
* 18 flight balanced sample: 9 `before` + 9 `after`.
* 16 channels used after dropping redundant ones: `volt2`, `amp2`, `E1 CHT2-4`, `E1 EGT2-4`.
* Sequence length truncated/padded to 1000 timesteps for fast pipeline test.
* Basic preprocessing inline: NaN fill with 0, RobustScaler per channel. Full preprocessing pipeline owned by feature engineering teammate.


### 0.1 Imports and Configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import dask.dataframe as dd
from sklearn.preprocessing import RobustScaler

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Sample configuration
N_PER_CLASS  = 9                 # 9 before + 9 after = 18 flights total
SEQ_LEN      = 1000              # truncate/pad to this length for fast pipeline test
N_CLASSES    = 2                 # binary: 0 = before, 1 = after
N_TRAIN, N_VAL, N_TEST = 12, 3, 3

# Channels to keep (drop redundant ones identified in EDA)
ALL_CHANNELS = [
    'volt1', 'volt2', 'amp1', 'amp2', 'FQtyL', 'FQtyR',
    'E1 FFlow', 'E1 OilT', 'E1 OilP', 'E1 RPM',
    'E1 CHT1', 'E1 CHT2', 'E1 CHT3', 'E1 CHT4',
    'E1 EGT1', 'E1 EGT2', 'E1 EGT3', 'E1 EGT4',
    'OAT', 'IAS', 'VSpd', 'NormAc', 'AltMSL'
]
DROP_CHANNELS = ['volt2', 'amp2', 'E1 CHT2', 'E1 CHT3', 'E1 CHT4',
                 'E1 EGT2', 'E1 EGT3', 'E1 EGT4']
KEEP_CHANNELS = [c for c in ALL_CHANNELS if c not in DROP_CHANNELS]
N_CHANNELS = len(KEEP_CHANNELS)

print(f'Channels kept ({N_CHANNELS}): {KEEP_CHANNELS}')

# Data paths (adjust if your Kaggle dataset path differs)
DATA_DIR = '/kaggle/input/aviation-maintenance-dataset-from-the-ngafid/all_flights/all_flights'
HEADER_CSV   = os.path.join(DATA_DIR, 'flight_header.csv')
PARQUET_PATH = os.path.join(DATA_DIR, 'one_parq')


### 0.2 Load Header and Pick 18 Flights

**Purpose:** Read the flight header CSV, filter to flights with valid `before` or `after` labels and sufficient length, then randomly pick 9 from each class with the same seed for reproducibility.


In [ ]:
# Load header
flight_header_df = pd.read_csv(HEADER_CSV, index_col='Master Index')
print(f'Total flights in header: {len(flight_header_df):,}')

# Filter: only before/after classes, drop very short flights
valid = flight_header_df[
    (flight_header_df['before_after'].isin(['before', 'after'])) &
    (flight_header_df['flight_length'] >= 100)  # skip suspicious short flights
].copy()
print(f'After filtering (before/after only, length >= 100): {len(valid):,}')

# Stratified random sample: 9 per class
rng = np.random.default_rng(SEED)
sampled_ids = []
for cls in ['before', 'after']:
    pool = valid.index[valid['before_after'] == cls].to_numpy()
    picked = rng.choice(pool, size=N_PER_CLASS, replace=False)
    sampled_ids.extend(picked)
sampled_ids = np.array(sorted(sampled_ids))

# Build label vector: 0 = before, 1 = after
sampled_header = flight_header_df.loc[sampled_ids].copy()
sampled_header['y'] = (sampled_header['before_after'] == 'after').astype(int)

print(f'\nSampled {len(sampled_ids)} flights:')
print(sampled_header['before_after'].value_counts())
print(f'\nMaster Indices: {sampled_ids.tolist()}')


### 0.3 Load Sensor Data via Dask

**Purpose:** Read the parquet sensor data, filter to only the 18 sampled flights in one scan, then split back into per flight arrays.


In [ ]:
# Lazy load full parquet
flight_data_df = dd.read_parquet(PARQUET_PATH)
print(f'Parquet partitions: {flight_data_df.npartitions}')
print(f'Sensor columns: {list(flight_data_df.columns)[:6]}...')

# Filter inside each partition to only sampled IDs
sampled_set = set(sampled_ids)
def keep_sampled(df):
    return df[df.index.isin(sampled_set)]

print('\nLoading 18 flights from parquet (this may take a minute)...')
sampled_pd = flight_data_df.map_partitions(keep_sampled).compute()
print(f'Loaded {len(sampled_pd):,} rows for {sampled_pd.index.nunique()} flights')


### 0.4 Build Tensors: X, Mask, y

**Purpose:** Convert per flight DataFrames into a single tensor of shape `(18, 1000, 16)`. Pad with zeros on the right if the flight is shorter than 1000 timesteps, truncate if longer. Build an attention mask marking real vs. padded positions.


In [ ]:
# Initialize containers
X = np.zeros((len(sampled_ids), SEQ_LEN, N_CHANNELS), dtype=np.float32)
mask = np.zeros((len(sampled_ids), SEQ_LEN), dtype=np.float32)
y = np.zeros(len(sampled_ids), dtype=np.int64)
master_indices = np.zeros(len(sampled_ids), dtype=np.int64)

for i, flight_id in enumerate(sampled_ids):
    # Extract this flight's data
    flight_df = sampled_pd.loc[[flight_id]] if isinstance(sampled_pd.loc[flight_id], pd.Series)                 else sampled_pd.loc[flight_id]
    if isinstance(flight_df, pd.Series):
        flight_df = flight_df.to_frame().T

    # Keep only desired channels, fill NaN with 0 (basic; full pipeline handles this better)
    arr = flight_df[KEEP_CHANNELS].to_numpy(dtype=np.float32)
    arr = np.nan_to_num(arr, nan=0.0)

    # Truncate or pad
    L = min(arr.shape[0], SEQ_LEN)
    X[i, :L] = arr[:L]
    mask[i, :L] = 1.0

    # Label and ID
    y[i] = sampled_header.loc[flight_id, 'y']
    master_indices[i] = flight_id

print(f'X shape : {X.shape}  (flights, timesteps, channels)')
print(f'mask shape : {mask.shape}')
print(f'y shape : {y.shape}  values={y.tolist()}')
print(f'\nReal length stats: min={mask.sum(axis=1).min():.0f}, '
      f'mean={mask.sum(axis=1).mean():.0f}, max={mask.sum(axis=1).max():.0f}')


### 0.5 Per Channel Scaling

**Purpose:** Apply RobustScaler (median and IQR) to each channel using only real (non padded) timesteps. The full pipeline should fit scalers on training data only, but for this 18 flight test we fit globally for simplicity.


In [ ]:
# Fit scaler on flattened real timesteps only
real_data = X[mask.astype(bool)]  # (n_real_steps, n_channels)
scaler = RobustScaler()
scaler.fit(real_data)

# Apply scaler to all timesteps (padded positions get scaled too, mask will ignore them)
X_flat = X.reshape(-1, N_CHANNELS)
X_scaled = scaler.transform(X_flat).reshape(X.shape).astype(np.float32)

# Re zero the padded positions after scaling
X_scaled = X_scaled * mask[..., np.newaxis]

print(f'X_scaled mean (real only): {X_scaled[mask.astype(bool)].mean():.4f}')
print(f'X_scaled std  (real only): {X_scaled[mask.astype(bool)].std():.4f}')
print(f'\nReady. Variables for downstream:')
print(f'  X_scaled       : {X_scaled.shape}')
print(f'  mask           : {mask.shape}')
print(f'  y              : {y.shape}')
print(f'  master_indices : {master_indices.shape}')

# Final tensor names matching the downstream pipeline
X_mock = X_scaled
mask_mock = mask
y_mock = y
N_TOTAL = len(y)


---
## Step 1. Data Preparation

### 1.1 Train / Val / Test Split

**Purpose:** Split the 18 flights into 12 train, 3 val, and 3 test. In the real pipeline this must be a **group split by Master Index** so flights from the same maintenance event do not leak across splits. For the mock test we use a stratified random split.


In [ ]:
from sklearn.model_selection import train_test_split

# First split off test (3 flights), then split remaining into train (12) and val (3)
idx_trainval, idx_test = train_test_split(
    np.arange(N_TOTAL), test_size=N_TEST, stratify=y_mock, random_state=SEED
)
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=N_VAL, stratify=y_mock[idx_trainval], random_state=SEED
)

print(f'Train indices: {idx_train.tolist()}  (n={len(idx_train)})')
print(f'Val indices  : {idx_val.tolist()}   (n={len(idx_val)})')
print(f'Test indices : {idx_test.tolist()}  (n={len(idx_test)})')
print(f'\nTrain class balance: {np.bincount(y_mock[idx_train])}')
print(f'Val   class balance: {np.bincount(y_mock[idx_val])}')
print(f'Test  class balance: {np.bincount(y_mock[idx_test])}')


### 1.2 PyTorch Dataset and DataLoader

**Purpose:** Wrap the arrays into a PyTorch `Dataset` so we can iterate in batches. Each sample returns the sensor tensor, the attention mask, and the label.


In [ ]:
from torch.utils.data import Dataset, DataLoader

class FlightDataset(Dataset):
    """Wraps preprocessed flight tensors for the Transformer."""

    def __init__(self, X, mask, y):
        self.X    = torch.from_numpy(X).float()
        self.mask = torch.from_numpy(mask).float()
        self.y    = torch.from_numpy(y).long()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, i):
        return self.X[i], self.mask[i], self.y[i]


BATCH_SIZE = 4  # small batch for the 18 flight mock test

train_ds = FlightDataset(X_mock[idx_train], mask_mock[idx_train], y_mock[idx_train])
val_ds   = FlightDataset(X_mock[idx_val],   mask_mock[idx_val],   y_mock[idx_val])
test_ds  = FlightDataset(X_mock[idx_test],  mask_mock[idx_test],  y_mock[idx_test])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# Sanity check one batch
xb, mb, yb = next(iter(train_loader))
print(f'Batch X    : {xb.shape}')
print(f'Batch mask : {mb.shape}')
print(f'Batch y    : {yb.shape}  values={yb.tolist()}')


---
## Step 2. Model Architecture

### 2.1 Positional Encoding

**Purpose:** Transformers have no built in notion of sequence order. We add a sinusoidal positional encoding so the model knows which timestep is which. Sinusoidal is parameter free and generalizes to sequence lengths unseen during training.


In [ ]:
import math
import torch.nn as nn

class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding from Vaswani et al. (2017)."""

    def __init__(self, d_model, max_len=10000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        return x + self.pe[:, :x.size(1)]


### 2.2 Transformer Classifier

**Purpose:** Defines the full model. The pipeline is: project 16 sensor channels into a `d_model` dimensional embedding, add positional encoding, pass through Transformer encoder layers, apply masked mean pooling to collapse the time dimension, then a linear layer produces 2 class logits.


In [ ]:
class FlightTransformer(nn.Module):
    def __init__(self, n_channels, d_model=64, nhead=4, num_layers=2,
                 dim_feedforward=128, dropout=0.1, n_classes=2, max_len=10000):
        super().__init__()
        # Project raw sensor channels into model dimension
        self.input_proj = nn.Linear(n_channels, d_model)
        self.pos_enc    = PositionalEncoding(d_model, max_len=max_len)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward, dropout=dropout,
            batch_first=True, activation='gelu'
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.norm    = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.head    = nn.Linear(d_model, n_classes)

    def forward(self, x, mask):
        # x: (B, T, C), mask: (B, T) where 1 means real, 0 means pad
        x = self.input_proj(x)         # (B, T, d_model)
        x = self.pos_enc(x)
        # PyTorch encoder expects key_padding_mask where True means ignore
        key_padding_mask = (mask == 0)
        x = self.encoder(x, src_key_padding_mask=key_padding_mask)
        x = self.norm(x)

        # Masked mean pooling over the time axis
        mask_exp = mask.unsqueeze(-1)                          # (B, T, 1)
        x_sum    = (x * mask_exp).sum(dim=1)                   # (B, d_model)
        x_mean   = x_sum / mask_exp.sum(dim=1).clamp(min=1e-6)

        x_mean = self.dropout(x_mean)
        return self.head(x_mean)                               # (B, n_classes)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FlightTransformer(
    n_channels=N_CHANNELS, d_model=64, nhead=4, num_layers=2,
    dim_feedforward=128, dropout=0.1, n_classes=N_CLASSES, max_len=SEQ_LEN
).to(device)

print(model)
print(f'\nTotal params: {sum(p.numel() for p in model.parameters()):,}')
print(f'Device      : {device}')


---
## Step 3. Training Setup

**Purpose:** Define the loss function, optimizer, learning rate scheduler, and evaluation metrics. Cross entropy with class weights handles any class imbalance. AdamW with warmup and cosine decay is the standard recipe for Transformers.


In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# Class weights for imbalance (computed from training labels)
class_counts = np.bincount(y_mock[idx_train], minlength=N_CLASSES)
class_weights = torch.tensor(
    len(idx_train) / (N_CLASSES * np.maximum(class_counts, 1)),
    dtype=torch.float32, device=device
)
print(f'Class counts (train): {class_counts}')
print(f'Class weights        : {class_weights.cpu().numpy()}')

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

N_EPOCHS  = 5  # small for the 18 flight mock test
scheduler = OneCycleLR(
    optimizer, max_lr=1e-3,
    steps_per_epoch=len(train_loader), epochs=N_EPOCHS,
    pct_start=0.1, anneal_strategy='cos'
)


### 3.1 Evaluation Metrics

**Purpose:** Helper function that computes accuracy, F1 macro, and per class recall on a given dataloader. F1 macro is the headline metric because it weights both classes equally regardless of imbalance.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for xb, mb, yb in loader:
        xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
        logits = model(xb, mb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_labels.extend(yb.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc      = accuracy_score(all_labels, all_preds)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    recall   = recall_score(all_labels, all_preds, average=None, zero_division=0)
    return {'loss': avg_loss, 'acc': acc, 'f1_macro': f1, 'recall_per_class': recall,
            'preds': all_preds, 'labels': all_labels}


---
## Step 4. Training Loop

**Purpose:** Iterate over epochs, track training and validation metrics, and save the best checkpoint based on validation F1. For the mock test we run only 5 epochs to confirm the pipeline executes end to end.


In [ ]:
import copy

history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
best_f1, best_state = -1.0, None

for epoch in range(1, N_EPOCHS + 1):
    # Train
    model.train()
    epoch_loss = 0.0
    for xb, mb, yb in train_loader:
        xb, mb, yb = xb.to(device), mb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb, mb)
        loss   = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        epoch_loss += loss.item() * xb.size(0)
    train_loss = epoch_loss / len(train_loader.dataset)

    # Validate
    val = evaluate(model, val_loader, criterion)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val['loss'])
    history['val_f1'].append(val['f1_macro'])

    # Checkpoint best
    if val['f1_macro'] > best_f1:
        best_f1 = val['f1_macro']
        best_state = copy.deepcopy(model.state_dict())
        flag = '  <-- best'
    else:
        flag = ''

    print(f'Epoch {epoch:2d}/{N_EPOCHS} | train_loss={train_loss:.4f} | '
          f'val_loss={val["loss"]:.4f} | val_f1={val["f1_macro"]:.4f}{flag}')

# Restore best weights for downstream evaluation
if best_state is not None:
    model.load_state_dict(best_state)
print(f'\nBest val F1: {best_f1:.4f}')


---
## Step 5. Evaluation and Analysis

### 5.1 Test Set Metrics

**Purpose:** Apply the best checkpoint to the held out test set and report final metrics. With only 3 test flights the numbers are not meaningful. This just confirms the eval pipeline works.


In [ ]:
test_result = evaluate(model, test_loader, criterion)
print(f'Test loss        : {test_result["loss"]:.4f}')
print(f'Test accuracy    : {test_result["acc"]:.4f}')
print(f'Test F1-macro    : {test_result["f1_macro"]:.4f}')
print(f'Per-class recall : {test_result["recall_per_class"]}')

cm = confusion_matrix(test_result['labels'], test_result['preds'], labels=list(range(N_CLASSES)))
print(f'\nConfusion matrix:\n{cm}')


### 5.2 Training Curves

**Purpose:** Visualize training and validation loss and F1 across epochs to spot overfitting or undertraining.


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train', marker='o')
axes[0].plot(history['val_loss'],   label='val',   marker='s')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['val_f1'], color='green', marker='o')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1-macro'); axes[1].set_title('Validation F1')
axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()


### 5.3 Attention Visualization (optional)

**Purpose:** Extract attention weights from the first encoder layer to see which timesteps the model attends to. Useful for interpretability but not required for the mock test.


In [ ]:
# Placeholder. Implement after the model is trained on real preprocessed data.
# Approach: register a forward hook on the first MultiheadAttention module,
# pass a single flight through the model, average attention weights across heads,
# and plot a (seq_len x seq_len) heatmap.
print('Attention visualization. Implement after real-data training.')


---
## Step 6. Ablation Studies (planning)

**Purpose:** Once the full data baseline is established, run controlled experiments to understand which design choices matter. Skip during mock test.

Planned ablations:

1. **Positional encoding:** sinusoidal vs. learned vs. none
2. **Pooling strategy:** masked mean vs. CLS token vs. attention pooling
3. **Channel reduction:** full 23 channels vs. reduced 16
4. **Sequence length:** truncate at p75 vs. p90 vs. p95
5. **Downsampling:** every 1s vs. every 5s vs. every 10s


---
## Step 7. Documentation and Handoff

**Purpose:** Save the trained model, preprocessing config, and key results so teammates can reproduce or load for inference.


In [ ]:
# Save best model state and config for handoff
import json
from pathlib import Path

OUT_DIR = Path('./transformer_artifacts')
OUT_DIR.mkdir(exist_ok=True)

torch.save(model.state_dict(), OUT_DIR / 'best_model.pt')

config = {
    'n_channels'     : N_CHANNELS,
    'kept_channels'  : KEEP_CHANNELS,
    'seq_len'        : SEQ_LEN,
    'n_classes'      : N_CLASSES,
    'd_model'        : 64,
    'nhead'          : 4,
    'num_layers'     : 2,
    'dim_feedforward': 128,
    'dropout'        : 0.1,
    'batch_size'     : BATCH_SIZE,
    'n_epochs'       : N_EPOCHS,
    'best_val_f1'    : float(best_f1),
    'test_f1'        : float(test_result['f1_macro']),
    'sampled_master_indices': master_indices.tolist(),
}
with open(OUT_DIR / 'config.json', 'w') as f:
    json.dump(config, f, indent=2)

print(f'Saved to {OUT_DIR.resolve()}/')
print(json.dumps(config, indent=2))


---
## Next Steps After Mock Test Passes

1. **Replace Step 0 (Assumption Block)** with the full preprocessed data loader from the feature engineering teammate.
2. **Increase `N_EPOCHS`** to 30 to 50 with early stopping on validation F1.
3. **Switch to group based split** by Master Index to prevent leakage across maintenance events.
4. **Tune hyperparameters.** `d_model` in {64, 128, 256}, `num_layers` in {2, 4, 6}.
5. **Add mixed precision training** (`torch.cuda.amp`) once full dataset is in play.
6. **Increase `SEQ_LEN`** back to 5000 or higher to capture more temporal detail.
7. **Run ablation studies** (Step 6).
8. **Compare against baselines.** XGBoost on per flight aggregates, LSTM.
